# Feedback Triage

Fetch thumbs-down feedback from Langfuse and create a human-reviewable triage CSV with Markdown trace files.

## Setup

Required environment variables:

- `LANGFUSE_BASE_URL` defaults to `https://cloud.langfuse.com`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` for complaint triage classification

In [5]:
import com.langfuse.client.LangfuseClient
import dev.example.langfuse.FeedbackTriageExporter
import dev.example.langfuse.LangfuseFeedbackClient
import dev.example.langfuse.LlmFeedbackTriager
import dev.dokimos.springai.SpringAiSupport
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.io.readCSV
import org.jetbrains.kotlinx.jupyter.api.HTML
import org.springframework.ai.chat.client.ChatClient
import org.springframework.ai.openai.OpenAiChatModel
import org.springframework.ai.openai.OpenAiChatOptions
import org.springframework.ai.openai.api.OpenAiApi
import java.nio.file.Files
import java.nio.file.Path

In [6]:
val langfuseBaseUrl = System.getenv("LANGFUSE_BASE_URL") ?: "https://cloud.langfuse.com"
val langfusePublicKey = requireNotNull(System.getenv("LANGFUSE_PUBLIC_KEY")) { "LANGFUSE_PUBLIC_KEY must be set" }
val langfuseSecretKey = requireNotNull(System.getenv("LANGFUSE_SECRET_KEY")) { "LANGFUSE_SECRET_KEY must be set" }
val openAiApiKey = requireNotNull(System.getenv("OPENAI_API_KEY")) { "OPENAI_API_KEY must be set" }

val langfuseClient = LangfuseClient.builder()
    .url(langfuseBaseUrl)
    .credentials(langfusePublicKey, langfuseSecretKey)
    .build()

val feedbackClient = LangfuseFeedbackClient(langfuseClient)

val openAiApi = OpenAiApi.builder().apiKey(openAiApiKey).build()
val triageChatModel = OpenAiChatModel.builder()
    .openAiApi(openAiApi)
    .defaultOptions(
        OpenAiChatOptions.builder()
            .model(OpenAiApi.ChatModel.GPT_5_CHAT_LATEST)
            .temperature(0.0)
            .build()
    )
    .build()
val triageJudge = SpringAiSupport.asJudge(ChatClient.builder(triageChatModel))
val feedbackTriager = LlmFeedbackTriager(triageJudge)
val triageExporter = FeedbackTriageExporter(feedbackClient, feedbackTriager)

## Export Negative Feedback

The CSV is the review index. The linked Markdown files contain full trace details.

In [7]:
val export = triageExporter.exportNegativeFeedback(
    limit = 20,
    outputDir = Path.of("eval-data/feedback-triage")
)

println("Exported ${export.exportedRows} rows")
println("CSV: ${export.csvFile.toAbsolutePath()}")
println("Traces: ${export.tracesDirectory.toAbsolutePath()}")

Exported 5 rows
CSV: /Users/urs/development/github/ai/kotlin-edd-talk/spring-ai/src/notebooks/eval-data/feedback-triage/negative-feedback-triage.csv
Traces: /Users/urs/development/github/ai/kotlin-edd-talk/spring-ai/src/notebooks/eval-data/feedback-triage/traces


## Review The CSV

The next two cells show the same CSV as a Kotlin DataFrame and as a custom HTML table.

In [8]:
val triageDf = DataFrame.readCSV(export.csvFile.toFile())
triageDf

traceId,observationId,sessionId,timestamp,rating,request,answer,userFeedbackReason,complaintSummary,failureMode,affectedCapability,severity,confidence,rationale,humanDecision,humanNotes,regressionCandidate,traceFile,langfuseUrl
659d9afa506e0c6127cc9bb6ef8f60ee,f5daf550a7079825,a006848b-59f8-49d7-83be-d8d721a3432e,2026-05-16T15:31:59.582Z,DOWN,Beginner sessions about KMP and AI.,Here are some **beginner-friendly Kot...,You suggested sessions in the past,The user felt the assistant repeated ...,content repetition or lack of novelty...,session recommendation,medium,"85,000000",The feedback explicitly mentions that...,null,null,null,traces/trace-659d9afa506e0c6127cc9bb6...,/project/cmhemduau010rad07o2h9gjn1/tr...
e15c492e9070f51df05cd202857efc04,e4761ca6315b6fa0,77004c07-83d3-4718-a795-69f58301b7fb,2026-05-16T15:30:11.375Z,DOWN,I'm interested in beginner session ab...,Here are some Kotlin Multiplatform (K...,Sessions recommendation of the past,The user says the assistant recommend...,outdated or stale content provided to...,session recommendation,medium,"95,000000",The feedback explicitly mentions 'Ses...,null,null,null,traces/trace-e15c492e9070f51df05cd202...,/project/cmhemduau010rad07o2h9gjn1/tr...
e267a9ef4360fd3cdd1d43daf2c03c22,379e6c0ed4f5f4fb,77004c07-83d3-4718-a795-69f58301b7fb,2026-05-16T15:27:08.671Z,DOWN,I'm interested in beginner session ab...,Here are some Kotlin Multiplatform (K...,null,The user gave negative feedback witho...,unclear dissatisfaction with relevanc...,session recommendation,low,"4,000000",The assistant response appears approp...,null,null,null,traces/trace-e267a9ef4360fd3cdd1d43da...,/project/cmhemduau010rad07o2h9gjn1/tr...
da80b24a8be2ad2aa1c26c507a2e19e5,93b040dada8a9e89,70f4cbae-59b3-4421-9a96-86764e33dac2,2026-04-09T13:33:31.036Z,DOWN,I don't like your tone,"Arrr, me apologies, matey! I meant no...",null,The user disliked the assistant's pla...,inappropriate tone or style for user ...,tone adaptation,medium,"95,000000",The user explicitly stated dissatisfa...,null,null,null,traces/trace-da80b24a8be2ad2aa1c26c50...,/project/cmhemduau010rad07o2h9gjn1/tr...
0f950ed4215e1e08cac9353c58af941f,d9c6dbcfcb4c953e,df2a54e0-bca8-4a59-b1d1-a539b6e813ac,2026-02-20T17:57:58.891Z,DOWN,I’m new to Kotlin—can you suggest 3 b...,Here are three beginner-friendly Kotl...,null,The user gave a negative rating witho...,unclear dissatisfaction or mismatch b...,session recommendation,low,"4,000000",The assistant’s answer appears accura...,null,null,null,traces/trace-0f950ed4215e1e08cac9353c...,null


## Single Record View

Set `selectedIndex` to inspect one feedback item top-down.

In [9]:
val selectedIndex = 0
val selectedRow = triageDf[selectedIndex]

val selectedRecord = dataFrameOf(
    "field" to triageDf.columnNames(),
    "value" to triageDf.columnNames().map { column ->
        selectedRow[column]?.toString().orEmpty()
    }
)

selectedRecord

field,value
traceId,659d9afa506e0c6127cc9bb6ef8f60ee
observationId,f5daf550a7079825
sessionId,a006848b-59f8-49d7-83be-d8d721a3432e
timestamp,2026-05-16T15:31:59.582Z
rating,DOWN
request,Beginner sessions about KMP and AI.
answer,Here are some **beginner-friendly Kot...
userFeedbackReason,You suggested sessions in the past
complaintSummary,The user felt the assistant repeated ...
failureMode,content repetition or lack of novelty...


## Human Review

Open `negative-feedback-triage.csv` and fill these fields manually:

- `humanDecision`: `ok` or `nok`
- `humanNotes`: short reviewer note
- `regressionCandidate`: `yes` for issues that should become evals

Use the `traceFile` column to inspect the full Markdown trace when needed.